In [1]:
# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# --- Load processed dataset ---
df = pd.read_csv("../data/processed_hr_dataset.csv")

# --- Select numeric features for clustering ---
numeric_features = ["Age", "DailyRate", "DistanceFromHome", "YearsAtCompany",
                    "YearsInCurrentRole", "TotalWorkingYears", "JobLevel",
                    "StockOptionLevel", "TrainingTimesLastYear", "NumCompaniesWorked",
                    "MonthlyIncome"]

X = df[numeric_features]

# --- Scale features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)



# --- Determine best number of clusters using silhouette score ---
sil_scores = []
K_range = range(2, 11)  # try 2 to 10 clusters

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"Silhouette score for k={k}: {score:.3f}")

# --- Plot silhouette scores ---
plt.figure(figsize=(6,4))
plt.plot(K_range, sil_scores, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs Number of Clusters")
plt.show()

# --- Select best k ---
best_k = K_range[np.argmax(sil_scores)]
print(f"\n✅ Best number of clusters based on silhouette score: {best_k}")

# --- Fit final KMeans ---
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)
df["Cluster"] = cluster_labels

# --- Cluster sizes and averages ---
print("\nCluster Sizes:")
display(df["Cluster"].value_counts().sort_index())

print("\nCluster Feature Averages:")
display(df.groupby("Cluster")[numeric_features].mean().round(2))

# --- PCA 2D visualization ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,5))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=cluster_labels, palette="tab10", s=60, alpha=0.7)
centroids_pca = PCA(n_components=2).fit_transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_pca[:,0], centroids_pca[:,1], c='black', s=150, marker='X', label='Centroids')
plt.title(f"Employee Clusters (k={best_k}) - PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

# --- Boxplots of key numeric features per cluster ---
key_features = ["Age", "YearsAtCompany", "MonthlyIncome"]
for feature in key_features:
    plt.figure(figsize=(6,4))
    sns.boxplot(x="Cluster", y=feature, hue="Cluster", data=df, palette="Set2",showfliers=False)
    plt.title(f"{feature} distribution by Cluster")
    plt.legend([],[], frameon=False)  # hide legend if you want
    plt.show()

print("\nCluster Interpretations:")

cluster_summary = df.groupby("Cluster")[numeric_features].mean().round(2)

for cluster in cluster_summary.index:
    desc = []
    row = cluster_summary.loc[cluster]
    
    # Age
    if row["Age"] > df["Age"].mean():
        desc.append("older employees")
    else:
        desc.append("younger employees")
    
    # Tenure
    if row["YearsAtCompany"] > df["YearsAtCompany"].mean():
        desc.append("long tenure")
    else:
        desc.append("short tenure")
    
    # MonthlyIncome
    if row["MonthlyIncome"] > df["MonthlyIncome"].mean():
        desc.append("high income")
    else:
        desc.append("low income")
    
    print(f"Cluster {cluster}: {', '.join(desc)}")


# --- Save clustering model and scaler ---
joblib.dump(kmeans_final, "../models/kmeans_clusters.joblib")
joblib.dump(scaler, "../models/scaler_for_clustering.joblib")
print("\nKMeans model and scaler saved for future use")


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed_hr_dataset.csv'